# Notebook 10: AlphaZero Curriculum Focus

Este notebook aprofunda uma agenda especifica de curriculum para `AlphaZero`, mantendo a comparacao global no notebook `09`.


## Objetivo

O objetivo deste notebook e:

- analisar em detalhe uma agenda escolhida;
- inspecionar curvas por seed;
- comparar checkpoints finais entre seeds e observar jogos de exemplo.


In [ ]:
from __future__ import annotations

import json
import statistics
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Could not find project root containing config.yaml")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / "outputs"
CURRICULUM_OUTPUTS = OUTPUTS / "alphazero_curriculum_experiments"
CURRICULUM_OUTPUTS.mkdir(parents=True, exist_ok=True)

from connect4_rl.agents.baselines import HeuristicAgent, RandomAgent
from connect4_rl.agents.learning import AlphaZeroConfig
from connect4_rl.envs.connect_four import apply_action, initial_state, is_terminal, legal_actions, render_ascii
from connect4_rl.experiments import build_agent_from_checkpoint, compute_elo_ratings, round_robin_detailed
from connect4_rl.experiments.alphazero_curriculum import (
    build_default_alphazero_curricula,
    expand_curriculum_schedule,
    train_alphazero_with_curriculum,
)
from connect4_rl.config import load_config

CONFIG = load_config(ROOT / "config.yaml")
NOTEBOOK_DEVICE = CONFIG.resolve_device()
if NOTEBOOK_DEVICE == "cuda":
    torch.set_default_device(NOTEBOOK_DEVICE)
print({"device": NOTEBOOK_DEVICE, "outputs": str(CURRICULUM_OUTPUTS)})


## Passo 1: Definicao da agenda em foco

Escolhe uma agenda e, opcionalmente, corre novas seeds so para essa familia.


In [ ]:
run_training = False

curricula = build_default_alphazero_curricula()
focus_agenda = "curriculum_probabilistic_bridge"
seeds = [7, 17, 27]

focus_config = AlphaZeroConfig(
    **{
        **asdict(CONFIG.alphazero),
        "episodes": 180,
        "mcts_simulations": 80,
        "eval_mcts_simulations": 120,
        "eval_interval": 30,
        "eval_games": 12,
        "n_res_blocks": 4,
        "updates_per_episode": 1,
        "replay_warmup_games": 12,
        "tactical_eval_examples": 64,
        "device": NOTEBOOK_DEVICE,
    }
)

focus_root = CURRICULUM_OUTPUTS
{
    "focus_agenda": focus_agenda,
    "seeds": seeds,
    "episodes": focus_config.episodes,
    "device": focus_config.device,
}


## Passo 2: Vista da agenda

Esta celula mostra exatamente quantos episodios cada fase ocupa.


In [ ]:
definition = curricula[focus_agenda]
schedule, phase_summary = expand_curriculum_schedule(focus_config.episodes, definition, seed=seeds[0])
print(definition.description)
for item in phase_summary:
    print(item)
print("counts:", {kind: schedule.count(kind) for kind in sorted(set(schedule))})


## Passo 3: Treino opcional da agenda em foco


In [ ]:
if run_training:
    for seed in seeds:
        run_config = AlphaZeroConfig(**{**asdict(focus_config), "seed": seed})
        checkpoint_dir = focus_root / f"{focus_agenda}_seed_{seed}"
        print(f"=== Running {focus_agenda} | seed={seed} ===")
        _agent, metrics = train_alphazero_with_curriculum(
            definition,
            run_config,
            checkpoint_dir=checkpoint_dir,
        )
        print("Last eval:", metrics.evaluation[-1] if metrics.evaluation else {})
else:
    print("Training skipped. Set run_training = True to launch this agenda.")


## Passo 4: Carregamento das corridas da agenda


In [ ]:
def load_focus_runs(root: Path, family: str) -> list[dict]:
    runs = []
    for metrics_path in sorted(root.glob(f"{family}_seed_*/metrics_final.json")):
        seed = int(metrics_path.parent.name.rsplit("_", 1)[-1])
        data = json.loads(metrics_path.read_text(encoding="utf-8"))
        runs.append({"seed": seed, "metrics_path": metrics_path, "data": data})
    return runs

focus_runs = load_focus_runs(focus_root, focus_agenda)
[(run["seed"], len(run["data"].get("evaluation", []))) for run in focus_runs]


## Passo 5: Curvas por seed

Aqui observamos se a agenda e estavel ou se depende demasiado da seed.


In [ ]:
if focus_runs:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.ravel()

    for run in focus_runs:
        seed = run["seed"]
        evaluations = run["data"].get("evaluation", [])
        tactical = run["data"].get("tactical_accuracy", [])
        if not evaluations:
            continue
        episodes = [item["episode"] for item in evaluations]
        axes[0].plot(episodes, [item["vs_random_win_rate"] for item in evaluations], marker="o", label=f"seed {seed}")
        axes[1].plot(episodes, [item["vs_heuristic_win_rate"] for item in evaluations], marker="o", label=f"seed {seed}")
        axes[2].plot(episodes, [item["vs_previous_win_rate"] for item in evaluations], marker="o", label=f"seed {seed}")
        if tactical:
            axes[3].plot([item["episode"] for item in tactical], [item["accuracy"] for item in tactical], marker="o", label=f"seed {seed}")

    axes[0].set_title("vs random")
    axes[1].set_title("vs heuristic")
    axes[2].set_title("vs previous snapshot")
    axes[3].set_title("tactical accuracy")
    for ax in axes:
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No runs found for this agenda.")


## Passo 6: Torneio simples entre seeds

Se existirem checkpoints validos, comparamos os melhores agentes de cada seed num round-robin pequeno.


In [ ]:
def build_seed_agents(runs: list[dict]) -> dict[str, object]:
    agents = {}
    for run in runs:
        checkpoint = run["data"].get("best_checkpoint_path")
        config = run["data"].get("config", {})
        if not checkpoint:
            continue
        agents[f"{focus_agenda}_seed_{run['seed']}"] = build_agent_from_checkpoint(
            "alphazero",
            ROOT / checkpoint,
            config,
            device=NOTEBOOK_DEVICE,
        )
    return agents

seed_agents = build_seed_agents(focus_runs)
list(seed_agents)


In [ ]:
if len(seed_agents) >= 2:
    match_log = round_robin_detailed(seed_agents, games_per_matchup=8)
    ratings = compute_elo_ratings(match_log)
    print("Elo:", ratings)
    match_log[:3]
else:
    print("Need at least two trained seeds with checkpoints.")


## Passo 7: Visualizacao de uma partida

A funcao abaixo ajuda a ver se o agente em foco esta realmente a jogar de forma mais estruturada.


In [ ]:
def play_and_render(agent, opponent, controlled_player: int = 1) -> str:
    state = initial_state()
    frames = [render_ascii(state)]
    move_idx = 0
    while not is_terminal(state):
        if state.current_player == controlled_player:
            action = agent.select_action(state, legal_actions(state))
            actor = agent.name
        else:
            action = opponent.select_action(state, legal_actions(state))
            actor = opponent.name
        state = apply_action(state, action)
        move_idx += 1
        frames.append(f"Move {move_idx}: {actor} played column {action}")
        frames.append(render_ascii(state))
    frames.append(f"Winner: {state.winner}")
    return "".join(frames)

if seed_agents:
    sample_name = sorted(seed_agents)[0]
    sample_agent = seed_agents[sample_name]
    print(sample_name)
    print(play_and_render(sample_agent, HeuristicAgent(seed=123), controlled_player=1))
else:
    print("No trained agent available yet.")


## Leituras esperadas

- se as curvas por seed forem muito instaveis, a agenda pode estar demasiado agressiva;
- se o torneio interno separar claramente uma seed das outras, vale a pena inspecionar esse checkpoint com mais cuidado;
- se os jogos renderizados continuarem a mostrar erros taticos basicos, o curriculum ainda nao esta a compensar as limitacoes do treino atual.
